In [24]:
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from xgboost.sklearn import XGBClassifier
from itertools import combinations
xgb_clf = XGBClassifier()

df = pd.read_csv(r'../data/results.csv')

df['date'] = pd.to_datetime(df['date'])
df = df[df['tournament'] != 'Friendly']

def winner(row):
    if(row['home_score'] > row['away_score']):
        return 'home'
    elif(row['home_score'] < row['away_score']):
        return 'away'
    else:
        return ('draw')

df['winner'] = df.apply(winner, axis=1)

home_df = pd.DataFrame({
    'date': df['date'],
    'team': df['home_team'],
    'won': df['winner'] == 'home',
    'goals_scored': df['home_score'],
    'goals_conceded': df['away_score']
})

away_df = pd.DataFrame({
    'date': df['date'],
    'team': df['away_team'],
    'won': df['winner'] == 'away',
    'goals_scored': df['away_score'],
    'goals_conceded': df['home_score']
})

win_result = pd.concat([home_df, away_df])
win_result = win_result.reset_index(drop=True)

win_rate = win_result.groupby('team')['won'].mean()
date_sorted = win_result.sort_values('date')
last_10 = date_sorted.groupby('team').tail(n=10)
win_rate_10 = last_10.groupby('team')['won'].mean()


goals = win_result.groupby('team')['goals_scored'].mean()
goals_allowed = win_result.groupby('team')['goals_conceded'].mean()
goal_differential = goals - goals_allowed


team_features = pd.concat([win_rate, win_rate_10, goals, goals_allowed, goal_differential], axis=1)
team_features.columns = ['overall_win_rate', 'recent_win_rate', 'goals_scored', 'goals_conceded', 'goal_differential']




def head_to_head(team1, team2):
    h2h = df[((df['home_team'] == team1) & (df['away_team'] == team2)) | ((df['away_team'] == team1) & (df['home_team'] == team2))]
    if(len(h2h) == 0):
        return 0.5
    team1_wins = len(h2h[((h2h['home_team'] == team1) & (h2h['winner'] == 'home')) |
                         ((h2h['away_team'] == team1) & (h2h['winner'] == 'away'))])
    return team1_wins / len(h2h)




match_data = df.merge(team_features, left_on='home_team', right_index=True)
match_data = match_data.merge(team_features, left_on='away_team', right_index=True, suffixes=('_home', '_away'))

def result(row):
    if(row['home_score'] > row['away_score']):
        return 1
    elif(row['home_score'] < row['away_score']):
        return -1
    else:
        return 0

match_data['result'] = match_data.apply(result, axis=1)

feature_cols = ['overall_win_rate_home', 'recent_win_rate_home', 'goals_scored_home', 'goals_conceded_home', 'goal_differential_home', 'overall_win_rate_away', 'recent_win_rate_away', 'goals_scored_away', 'goals_conceded_away', 'goal_differential_away']
X = match_data[feature_cols]
y = match_data['result']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)


le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

xgb_clf.fit(X_train, y_train_encoded)
y_pred = xgb_clf.predict(X_test)
accuracy = accuracy_score(y_test_encoded, y_pred)

def predict_winner(team1, team2):
    features1 = team_features.loc[team1]
    features1.index = [f + '_home' for f in features1.index]
    features2 = team_features.loc[team2]
    features2.index = [f + '_away' for f in features2.index]
    combined_features = pd.concat([features1, features2])
    prediction = xgb_clf.predict(combined_features.values.reshape(1,-1))
    if prediction == 2:
        return team1
    elif prediction == 0:
        return team2
    else:
        return "draw"

print(predict_winner("Brazil", "Argentina"))
print(predict_winner("France", "Germany"))
print(predict_winner("England", "Spain"))

print('Haiti' in team_features.index)
print('Curaçao' in team_features.index)
print('Jordan' in team_features.index)

groups = {
    "Group A": ["Mexico", "South Africa", "South Korea", "Denmark"],
    "Group B": ["Canada", "Greece", "Qatar", "Switzerland"],
    "Group C": ["Brazil", "Morocco", "Haiti", "Scotland"],
    "Group D": ["United States", "Paraguay", "Australia", "Ukraine"],
    "Group E": ["Germany", "Curaçao", "Ivory Coast", "Ecuador"],
    "Group F": ["Netherlands", "Japan", "Slovenia", "Tunisia"],
    "Group G": ["Belgium", "Egypt", "Iran", "New Zealand"],
    "Group H": ["Spain", "Cape Verde", "Saudi Arabia", "Uruguay"],
    "Group I": ["France", "Senegal", "Venezuela", "Norway"],
    "Group J": ["Argentina", "Algeria", "Austria", "Jordan"],
    "Group K": ["Portugal", "Indonesia", "Uzbekistan", "Colombia"],
    "Group L": ["England", "Croatia", "Ghana", "Panama"]
}

def simulate_group(teams):
    points = {team: 0 for team in teams}
    for team1, team2 in combinations(teams, 2):
        predict = predict_winner(team1, team2)
        if predict == team1:
            points[team1]+=3
        elif predict == team2:
            points[team2] += 3
        else:
            points[team1]+=1
            points[team2]+=1
    sorted_data_desc = dict(sorted(points.items(), key=lambda item: item[1], reverse=True))
    teams_list = list(sorted_data_desc.keys())
    points_list = list(sorted_data_desc.values())
    return teams_list[:2], (teams_list[2], points_list[2])

qualifiers = []
third_place_teams = []
for group_name, teams in groups.items():
    top_2, third = simulate_group(teams)
    qualifiers.extend(top_2)
    third_place_teams.append(third)
print("Qualifiers: ", qualifiers)
print("Third place teams: ", third_place_teams)





Brazil
France
Spain
True
True
True
Qualifiers:  ['Mexico', 'South Korea', 'Canada', 'Greece', 'Brazil', 'Morocco', 'United States', 'Australia', 'Germany', 'Ivory Coast', 'Netherlands', 'Japan', 'Iran', 'Belgium', 'Spain', 'Uruguay', 'France', 'Senegal', 'Argentina', 'Algeria', 'Portugal', 'Uzbekistan', 'England', 'Croatia']
Third place teams:  [('South Africa', 2), ('Qatar', 3), ('Scotland', 3), ('Ukraine', 3), ('Curaçao', 3), ('Slovenia', 2), ('Egypt', 3), ('Cape Verde', 1), ('Norway', 4), ('Austria', 3), ('Colombia', 3), ('Ghana', 3)]
